# تست کوچک: Augmentation فیزیکی با Point Group Symmetry روی یک ماده

## هدف
قبل از اعمال augmentation روی کل دیتاست (۳۵۸ ماده × ۲۴ عملیات تقارن)، باید روی یک ماده‌ی
نمونه تأیید کنیم که:
1. هر یک از ۲۴ عملیات تقارن (rotation + translation) واقعاً موقعیت اتم‌ها را به یک
   جابه‌جایی معتبر از همان مجموعه‌ی اتم‌ها نگاشت می‌کند (نه چیزی نامعتبر)
2. وقتی همین تقارن روی ماتریس IFC هم اعمال می‌شود، فرکانس‌های فونونی محاسبه‌شده با IFC
   transform‌شده (با موتور واقعی Phonopy) **دقیقاً همان فرکانس‌های اصلی** باشند — چون فونون
   یک کمیت فیزیکی نامتغیر زیر تقارن کریستال است؛ اگر این تست رد شود یعنی فرمول transform
   اشتباه است.

## ایده‌ی ریاضی Transform
برای یک عملیات تقارن با ماتریس چرخش `R` (روی fractional coordinates عمل می‌کند) و بردار
انتقال `t`:
- موقعیت اتم جدید: `frac_new = R @ frac_old + t` (با wrap به [0,1))
- لاتیس (در حالت کلی) ثابت می‌ماند چون این عملیات‌ها متعلق به همان گروه فضایی لاتیس هستند
- IFC: چون `IFC[i,j]` رابطه‌ی نیرو بین اتم `i` و `j` را توصیف می‌کند، زیر تقارن باید
  اتم‌ها هم با همان permutation که موقعیت‌شان جابه‌جا شده، در IFC هم جابه‌جا شوند، و علاوه
  بر آن خودِ تانسور `3×3` هر بلوک باید با ماتریس چرخش کارتزی (نه fractional) conjugate شود:
  `IFC_new[i', j'] = R_cart @ IFC[i, j] @ R_cart^T`
  که `i', j'` اتم‌های متناظر بعد از permutation هستند و `R_cart` نسخه‌ی کارتزی چرخش است.

In [ ]:
!pip install -q phonopy -t /kaggle/working/custom_lib
import sys
sys.path.append('/kaggle/working/custom_lib')
print("Installed")

In [8]:
import numpy as np
import yaml
from pathlib import Path
from phonopy import Phonopy
from phonopy.structure.atoms import PhonopyAtoms
from phonopy.file_IO import parse_FORCE_CONSTANTS
import spglib

FC_DIR     = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_force_constant_All/extracted_force_constant_C'
BANDS_DIR  = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_bands_C/extracted_bands_C'

FC_DIR_PATH = Path(FC_DIR)
BANDS_DIR_PATH = Path(BANDS_DIR)

fc_files = {f.stem: f for f in FC_DIR_PATH.glob('*.fc')}
band_yaml_files = {f.stem: f for f in BANDS_DIR_PATH.glob('*.yaml')}

common = sorted(set(fc_files) & set(band_yaml_files))
test_formula = common[0]
print(f"Test material: {test_formula}")

Test material: Cr2AlC


## بازسازی ماده با موتور واقعی Phonopy (دقیقاً مثل notebook های قبلی)

In [9]:
def find_correct_supercell_dim(unitcell, fc, q_test, real_freqs_test, n2_expected):
    n_images = n2_expected // len(unitcell.symbols)
    candidates = []
    for a in range(1, n_images + 1):
        if n_images % a != 0:
            continue
        rem = n_images // a
        for b in range(1, rem + 1):
            if rem % b != 0:
                continue
            c = rem // b
            candidates.append((a, b, c))

    best_dim, best_err = None, np.inf
    for (a, b, c) in candidates:
        try:
            phonon = Phonopy(unitcell, supercell_matrix=[[a,0,0],[0,b,0],[0,0,c]])
            if len(phonon.supercell.symbols) != n2_expected:
                continue
            phonon.force_constants = fc
            phonon.run_qpoints([q_test])
            freqs = np.sort(phonon.get_qpoints_dict()['frequencies'][0])
            if len(freqs) != len(real_freqs_test):
                continue
            err = np.max(np.abs(freqs - real_freqs_test))
            if err < best_err:
                best_err = err
                best_dim = (a, b, c)
        except Exception:
            continue
    return best_dim, best_err


with open(band_yaml_files[test_formula]) as f:
    data = yaml.safe_load(f)

lattice = np.array(data['lattice'])
symbols = [p['symbol'] for p in data['points']]
frac_coords = np.array([p['coordinates'] for p in data['points']])
n_atoms = len(symbols)

unitcell = PhonopyAtoms(symbols=symbols, cell=lattice, scaled_positions=frac_coords)

fc = parse_FORCE_CONSTANTS(str(fc_files[test_formula]))
n_supercell = fc.shape[1]

segment_len = data['segment_nqpoint'][0]
q_idx = min(segment_len // 2, len(data['phonon']) - 1)
q_test = data['phonon'][q_idx]['q-position']
real_freqs_test = np.sort([b['frequency'] for b in data['phonon'][q_idx]['band']])

dim, err = find_correct_supercell_dim(unitcell, fc, q_test, real_freqs_test, n_supercell)
print(f"Discovered supercell_dim: {dim} (match error: {err:.6f})")

phonon = Phonopy(unitcell, supercell_matrix=[[dim[0],0,0],[0,dim[1],0],[0,0,dim[2]]])
phonon.force_constants = fc

phonon.run_qpoints([[0, 0, 0]])
original_freqs = np.sort(phonon.get_qpoints_dict()['frequencies'][0])
print(f"Original frequencies at Gamma (THz): {original_freqs}")

/tmp/ipykernel_58/2068968917.py:22: DeprecationWarning: get_qpoints_dict() is deprecated. Use the qpoints property to access the Qpoints result object; its frequencies, eigenvectors, group_velocities, and dynamical_matrices attributes replace the dict keys.
  freqs = np.sort(phonon.get_qpoints_dict()['frequencies'][0])
/kaggle/working/custom_lib/phonopy/api_phonopy.py:278: UserWarning: Warning: Point group symmetries of supercell and primitivecell are different.
  self._search_primitive_symmetry()


Discovered supercell_dim: (3, 3, 1) (match error: 0.000000)
Original frequencies at Gamma (THz): [-3.07950122e-03  2.43763264e-03  2.43763266e-03  2.99455968e+00
  2.99455968e+00  4.77698563e+00  4.77698563e+00  4.88722860e+00
  5.60689545e+00  5.60689545e+00  7.92996353e+00  7.99231856e+00
  7.99231856e+00  8.11675096e+00  8.11675096e+00  1.01656150e+01
  1.07428656e+01  1.14309913e+01  2.03112874e+01  2.03112874e+01
  2.03589956e+01  2.03589956e+01  2.04610455e+01  2.04984014e+01]


/tmp/ipykernel_58/2068968917.py:59: DeprecationWarning: get_qpoints_dict() is deprecated. Use the qpoints property to access the Qpoints result object; its frequencies, eigenvectors, group_velocities, and dynamical_matrices attributes replace the dict keys.
  original_freqs = np.sort(phonon.get_qpoints_dict()['frequencies'][0])


## کشف عملیات تقارن با spglib

In [10]:
cell_tuple = (unitcell.cell, unitcell.scaled_positions, unitcell.numbers)
dataset = spglib.get_symmetry_dataset(cell_tuple, symprec=1e-3)

n_ops = len(dataset.rotations)
print(f"Space group: {dataset.international} (number {dataset.number})")
print(f"Number of symmetry operations: {n_ops}")

Space group: P6_3/mmc (number 194)
Number of symmetry operations: 24


## تابع تست: اعمال یک عملیات تقارن و بررسی نگه‌داری permutation اتم‌ها

برای هر عملیات `(R, t)`، موقعیت جدید هر اتم را حساب می‌کنیم و چک می‌کنیم که آیا این موقعیت
(با تلورانس عددی) با موقعیت یکی از اتم‌های اصلی (با همان نوع عنصر) یکی می‌شود — این
permutation لازم است تا بعداً IFC را هم به‌درستی جابه‌جا کنیم.

In [11]:
def find_atom_permutation(frac_coords, symbols, R, t, symprec=1e-3):
    """
    Apply (R, t) to all fractional coordinates and find which original atom
    each transformed position maps to (same element, matching position mod 1).
    Returns permutation array perm such that transformed atom i == original atom perm[i],
    or None if no valid permutation is found (transform invalid for this structure).
    """
    n = len(frac_coords)
    transformed = (frac_coords @ R.T + t) % 1.0
    perm = np.full(n, -1, dtype=int)

    for i in range(n):
        best_j, best_dist = -1, np.inf
        for j in range(n):
            if symbols[j] != symbols[i]:
                continue
            diff = transformed[i] - frac_coords[j]
            diff = diff - np.round(diff)  # wrap to nearest periodic image
            dist = np.linalg.norm(diff)
            if dist < best_dist:
                best_dist = dist
                best_j = j
        if best_dist > symprec:
            return None
        perm[i] = best_j

    if len(set(perm.tolist())) != n:
        return None  # not a valid bijection
    return perm

print("find_atom_permutation ready")

find_atom_permutation ready


In [12]:
valid_count = 0
invalid_count = 0

for op_idx in range(n_ops):
    R = dataset.rotations[op_idx]
    t = dataset.translations[op_idx]
    perm = find_atom_permutation(frac_coords, symbols, R, t)
    if perm is None:
        invalid_count += 1
        print(f"  op {op_idx}: INVALID (no consistent atom permutation found)")
    else:
        valid_count += 1

print(f"\nValid operations: {valid_count} / {n_ops}")
print(f"Invalid operations: {invalid_count} / {n_ops}")


Valid operations: 24 / 24
Invalid operations: 0 / 24


## تابع تست اصلی: Transform کامل (لاتیس بدون تغییر + IFC) و تأیید با Phonopy

برای یک عملیات تقارن مشخص، IFC را transform می‌کنیم و فرکانس فونون آن را با Phonopy واقعی
محاسبه می‌کنیم — باید دقیقاً با فرکانس‌های اصلی یکی باشد.

In [13]:
def transform_ifc(fc, perm, R_cart, n_atoms, n_supercell):
    """
    Transform the IFC tensor (n_atoms, n_supercell, 3, 3) under a symmetry operation.
    perm: permutation mapping transformed unit-cell atom index -> original atom index
    R_cart: 3x3 cartesian rotation matrix
    NOTE: this only permutes the unit-cell (reference) axis; the supercell axis ordering
    is assumed unchanged here since we are only testing the unit-cell block (Gamma-point
    sanity check), not the full supercell remapping.
    """
    # IFC_new[i, :, :, :] = R_cart @ IFC[perm[i], :, :, :] @ R_cart^T, applied to each 3x3 block
    fc_permuted = fc[perm]  # reorder reference-atom axis
    fc_new = np.einsum('ab,nscd,db->nsac', R_cart, fc_permuted, R_cart)
    # NOTE: this einsum applies R_cart @ block @ R_cart^T per (n,s) pair via index contraction
    return fc_new


def cartesian_rotation(R_frac, lattice):
    """Convert a fractional-coordinate rotation matrix to cartesian: R_cart = L^T @ R_frac @ L^-T"""
    L = lattice  # rows are lattice vectors
    R_cart = L.T @ R_frac @ np.linalg.inv(L.T)
    return R_cart


# pick the first non-identity, valid operation for a clean test
test_op_idx = None
for op_idx in range(1, n_ops):
    R = dataset.rotations[op_idx]
    t = dataset.translations[op_idx]
    perm = find_atom_permutation(frac_coords, symbols, R, t)
    if perm is not None:
        test_op_idx = op_idx
        break

print(f"Testing with symmetry operation index: {test_op_idx}")
R = dataset.rotations[test_op_idx]
t = dataset.translations[test_op_idx]
perm = find_atom_permutation(frac_coords, symbols, R, t)
print(f"Rotation matrix (fractional):\n{R}")
print(f"Translation: {t}")
print(f"Atom permutation: {perm}")

R_cart = cartesian_rotation(R, lattice)
print(f"\nCartesian rotation matrix:\n{R_cart}")
print(f"Determinant (should be +-1 for a valid rotation/reflection): {np.linalg.det(R_cart):.4f}")

Testing with symmetry operation index: 1
Rotation matrix (fractional):
[[-1  0  0]
 [ 0 -1  0]
 [ 0  0 -1]]
Translation: [-1.58761893e-14 -1.57651669e-14  0.00000000e+00]
Atom permutation: [3 2 1 0 5 4 6 7]

Cartesian rotation matrix:
[[-1.00000000e+00 -3.53262967e-17  0.00000000e+00]
 [ 0.00000000e+00 -1.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00 -1.00000000e+00]]
Determinant (should be +-1 for a valid rotation/reflection): -1.0000


In [14]:
fc_transformed = transform_ifc(fc, perm, R_cart, n_atoms, n_supercell)
print(f"Transformed IFC shape: {fc_transformed.shape} (should match original: {fc.shape})")

phonon_transformed = Phonopy(unitcell, supercell_matrix=[[dim[0],0,0],[0,dim[1],0],[0,0,dim[2]]])
phonon_transformed.force_constants = fc_transformed
phonon_transformed.run_qpoints([[0, 0, 0]])
transformed_freqs = np.sort(phonon_transformed.get_qpoints_dict()['frequencies'][0])

print(f"\nOriginal frequencies at Gamma (THz):    {original_freqs}")
print(f"Transformed frequencies at Gamma (THz): {transformed_freqs}")
print(f"\nMax absolute difference: {np.max(np.abs(original_freqs - transformed_freqs)):.8f} THz")

Transformed IFC shape: (8, 72, 3, 3) (should match original: (8, 72, 3, 3))

Original frequencies at Gamma (THz):    [-3.07950122e-03  2.43763264e-03  2.43763266e-03  2.99455968e+00
  2.99455968e+00  4.77698563e+00  4.77698563e+00  4.88722860e+00
  5.60689545e+00  5.60689545e+00  7.92996353e+00  7.99231856e+00
  7.99231856e+00  8.11675096e+00  8.11675096e+00  1.01656150e+01
  1.07428656e+01  1.14309913e+01  2.03112874e+01  2.03112874e+01
  2.03589956e+01  2.03589956e+01  2.04610455e+01  2.04984014e+01]
Transformed frequencies at Gamma (THz): [-1.14309913e+01 -1.07428656e+01 -8.11675096e+00 -8.11675096e+00
 -7.99231856e+00 -7.99231856e+00 -7.92996353e+00 -4.77698563e+00
 -4.77698563e+00 -3.07950122e-03  2.43763265e-03  2.43763265e-03
  2.99455968e+00  2.99455968e+00  4.88722860e+00  5.60689545e+00
  5.60689545e+00  1.01656150e+01  2.03112874e+01  2.03112874e+01
  2.03589956e+01  2.03589956e+01  2.04610455e+01  2.04984014e+01]

Max absolute difference: 12.76930419 THz


/tmp/ipykernel_58/280229360.py:7: DeprecationWarning: get_qpoints_dict() is deprecated. Use the qpoints property to access the Qpoints result object; its frequencies, eigenvectors, group_velocities, and dynamical_matrices attributes replace the dict keys.
  transformed_freqs = np.sort(phonon_transformed.get_qpoints_dict()['frequencies'][0])


## نتیجه‌گیری این تست

اگر `Max absolute difference` خیلی نزدیک به صفر باشد (مثلاً کمتر از `1e-4`)، یعنی فرمول
transform (هم permutation اتم‌ها و هم conjugate کردن تانسور IFC با چرخش کارتزی) درست است
و فیزیک واقعاً زیر این تقارن نامتغیر مانده — یعنی می‌توانیم با اطمینان این روش را روی کل
دیتاست (۳۵۸ ماده × تا ۲۴ عملیات) اعمال کنیم تا داده‌ی augment‌شده‌ی معتبر بسازیم.

اگر خطا بزرگ بود، باید فرمول transform (به‌خصوص نگاشت محور سوپرسل، که در این تست ساده‌شده
نادیده گرفته شد) را بازبینی کنیم.

**لطفاً کل خروجی این نوت‌بوک را برای من بفرستید.**